# Lab: the two-stage model under ocean forcing

The ice-stream lab measured the response of a full stress-balance model; {doc}`reduced-models` compressed that behavior into two ordinary differential equations. This lab integrates those two equations directly. The reward for giving up the stress balance is speed; runs that took minutes in icepack take milliseconds here, so questions that need thousands of runs, sensitivity sweeps, stability maps, and the statistics of a noisy ocean, become affordable. The lab closes with the experiment the reduced model exists for, fifty thousand years of stochastic ocean forcing and the grounding-line variability it produces, the null hypothesis against which observed retreat must be judged.

Everything here is plain numpy; no icepack is required.

## The model

State variables are the interior thickness $H$ and grounding-line position $L$, with

$$
\frac{dL}{dt} = \frac{Q - Q_g}{h_g}, \qquad
\frac{dH}{dt} = P - \frac{Q_g}{L} - \frac{H}{h_g L}\left(Q - Q_g\right),
$$

interior flux $Q = \nu H^{\alpha}/L^{\gamma}$, grounding-zone flux $Q_g = \Omega h_g^{\beta}$, and flotation thickness $h_g = -\lambda b(L)$, exactly as derived in {doc}`reduced-models`. We choose the steady state first, a grounding line at 400 km on a prograde bed with accumulation 0.3 m/yr, and derive the flux prefactors from the steady-state balance $PL = Q = Q_g$, which guarantees the model starts in equilibrium.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

lam = 1.1          # rho_w / rho_i
n_glen, m_fric = 3.0, 1.0/3.0
alpha, gamma = 2*n_glen + 2, n_glen          # interior-flux exponents (SIA)
beta = (m_fric + n_glen + 3) / (m_fric + 1)  # ~4.75, Schoof grounding-line exponent

def make_bed(b0=-100.0, bx=-1.0e-3):
    """Linear bed elevation b(x) = b0 + bx*x  (bx<0: deepens seaward, prograde)."""
    return (lambda x: b0 + bx*x), (lambda x: bx*np.ones_like(np.atleast_1d(x))[0] if np.isscalar(x) else bx*np.ones_like(x))

bed, bed_slope = make_bed()

# Choose the steady state, then derive the prefactors
P = 0.3            # m/yr
L0, H0 = 400e3, 1500.0
hg0 = -lam * bed(L0)
Omega = P * L0 / hg0**beta
nu = P * L0**(gamma + 1) / H0**alpha

def fluxes(H, L, Om):
    hg = -lam * bed(L)
    Q = nu * H**alpha / L**gamma
    Qg = Om * hg**beta
    return Q, Qg, hg

def rhs(state, Om):
    H, L = state
    Q, Qg, hg = fluxes(H, L, Om)
    dL = (Q - Qg) / hg
    dH = P - Qg / L - H * (Q - Qg) / (hg * L)
    return np.array([dH, dL])

def rk4(state, Om, dt):
    k1 = rhs(state, Om)
    k2 = rhs(state + 0.5*dt*k1, Om)
    k3 = rhs(state + 0.5*dt*k2, Om)
    k4 = rhs(state + dt*k3, Om)
    return state + dt*(k1 + 2*k2 + 2*k3 + k4)/6

Q, Qg, hg = fluxes(H0, L0, Omega)
print(f"steady-state check: PL = {P*L0:.4g}, Q = {Q:.4g}, Qg = {Qg:.4g} m^2/yr")

## Task 1: step response and the two timescales

Increase $\Omega$ by twenty percent at $t=0$, the reduced-model version of the melt forcing applied in the icepack lab, and integrate for five thousand years. Plot $H(t)$ and $L(t)$ on shared time axes. Then compute the eigenvalues of the numerical Jacobian at the original steady state and compare their inverse magnitudes against the two phases you see in the trajectories.

In [ ]:
dt = 0.5
years = 5000
nsteps = int(years / dt)

state = np.array([H0, L0])
Om_forced = 1.2 * Omega
traj = np.empty((nsteps, 2))
for i in range(nsteps):
    state = rk4(state, Om_forced, dt)
    traj[i] = state
t = dt * np.arange(1, nsteps + 1)

fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
axes[0].plot(t, traj[:, 0]); axes[0].set_ylabel("H (m)")
axes[1].plot(t, traj[:, 1] / 1e3); axes[1].set_ylabel("L (km)")
axes[1].set_xlabel("time (yr)")
fig.suptitle("Step response to a 20% increase in $\\Omega$");

In [ ]:
def jacobian(state, Om, eps=1e-4):
    J = np.zeros((2, 2))
    f0 = rhs(state, Om)
    for k in range(2):
        ds = state.copy()
        ds[k] *= (1 + eps)
        J[:, k] = (rhs(ds, Om) - f0) / (state[k] * eps)
    return J

evals = np.linalg.eigvals(jacobian(np.array([H0, L0]), Omega))
print("eigenvalues (1/yr):", evals)
print("e-folding times (yr):", np.sort(-1 / evals.real))

## Task 2: the stability number and critical slowing

Sweep the bed slope from strongly prograde ($b_x = -2\times10^{-3}$) toward and past the critical value, recomputing the steady state and the stability number

$$
\Sigma = \beta\,\frac{\bar L\, h_g'(\bar L)}{\bar h_g} - 1, \qquad h_g' = -\lambda\, b_x,
$$

at each slope. Plot the response gain $1/\Sigma$ and the slow e-folding time against $b_x$. Both should diverge as $\Sigma \to 0$, the critical slowing derived in {doc}`reduced-models`; mark the slope at which $\Sigma = 0$ and compare it with $h_g/(\beta L)$. What does the divergence imply for the trustworthiness of sensitivity estimates made near a flat bed?

In [ ]:
slopes = np.linspace(-2.0e-3, -0.05e-3, 40)
Sig, tau_slow = [], []
for bx in slopes:
    bed, bed_slope = make_bed(bx=bx)
    hg_s = -lam * bed(L0)
    Om_s = P * L0 / hg_s**beta          # re-derive prefactor so (H0, L0) stays steady
    Sigma = beta * L0 * (-lam * bx) / hg_s - 1
    ev = np.linalg.eigvals(jacobian(np.array([H0, L0]), Om_s))
    Sig.append(Sigma)
    tau_slow.append(np.max(-1 / ev.real))
bed, bed_slope = make_bed()   # restore default

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(slopes * 1e3, 1 / np.array(Sig))
axes[0].set_xlabel("bed slope (m/km)"); axes[0].set_ylabel("gain $1/\\Sigma$")
axes[1].semilogy(slopes * 1e3, tau_slow)
axes[1].set_xlabel("bed slope (m/km)"); axes[1].set_ylabel("slow e-folding time (yr)");

## Task 3: fifty thousand years of ocean noise

Now the experiment the model exists for. Hold the climate statistically steady, $\Omega(t) = \bar\Omega\,(1 + \sigma\,\xi_t)$ with $\xi_t$ independent standard-normal draws each year and $\sigma = 0.2$, a modest ocean variability with no trend whatever. Integrate for 50,000 years, plot $L(t)$, and compute its standard deviation and decorrelation time. Then answer, in writing, with reference to the resonant-filter discussion in {doc}`reduced-models` and the Hasselmann argument of {doc}`climate-forcing`:

1. How large are the grounding-line excursions, and over what timescales do they unfold, given that the forcing has zero memory from year to year?
2. A satellite record shows fifteen years of steady grounding-line retreat at a glacier on a prograde bed. Using your simulated null distribution, how strong is that evidence of forced change?
3. Repeat the run with the grounding line started just upstream of a retrograde reach (rebuild the bed with a bump, or simply flip the slope sign locally). How does noise interact with the instability threshold, and what does that mean for the deep-uncertainty tail of {doc}`sea-level`?

In [ ]:
rng = np.random.default_rng(7)
years = 50_000
dt = 0.5
steps_per_year = int(1 / dt)
sigma = 0.2

state = np.array([H0, L0])
L_series = np.empty(years)
for yr in range(years):
    Om_t = Omega * (1 + sigma * rng.standard_normal())
    for _ in range(steps_per_year):
        state = rk4(state, max(Om_t, 0.1 * Omega), dt)
    L_series[yr] = state[1]

ty = np.arange(years)
fig, axes = plt.subplots(figsize=(9, 3.5))
axes.plot(ty / 1e3, (L_series - L0) / 1e3, lw=0.6)
axes.set_xlabel("time (kyr)"); axes.set_ylabel("grounding-line anomaly (km)")
print(f"std of L: {np.std(L_series)/1e3:.2f} km")
ac = np.correlate(L_series - L_series.mean(), L_series - L_series.mean(), "full")[years-1:]
ac /= ac[0]
print(f"decorrelation time (1/e): {np.argmax(ac < 1/np.e)} yr")

## Task 4: synthesis

Set your three results side by side with the icepack lab. The full model and the reduced model agree on the phenomenology, fast partial adjustment, slow completion, runaway on retrograde slopes, and the reduced model adds what the full model cannot affordably provide, the statistics. Write a short paragraph stating, for a glacier you choose, which questions you would put to each tool and why; then state the one ingredient of the icepack experiment that the two-stage model cannot represent at all, and what observable difference you would expect it to make.